# 2026 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/nlp_project/data')


In [ ]:
import json

# read data
with open(DATA_DIR / 'train-claims.json') as f:
    train_claims = json.load(f)

with open(DATA_DIR / 'dev-claims.json') as f:
    dev_claims = json.load(f)

with open(DATA_DIR / 'test-claims-unlabelled.json') as f:
    test_claims = json.load(f)

with open(DATA_DIR / 'evidence.json') as f:
    evidence = json.load(f)

In [ ]:
print(f"train: {len(train_claims)}")
print(f"dev: {len(dev_claims)}")
print(f"test: {len(test_claims)}")
print(f"evidence: {len(evidence)}")

train: 1228
dev: 154
test: 153
evidence: 1208827


In [ ]:
from collections import Counter
import numpy as np

# 统计训练集标签分布
label_dist = Counter(c['claim_label'] for c in train_claims.values())
print('训练集标签分布:')
for lbl, cnt in label_dist.most_common():
    print(f'    {lbl:20s} {cnt:>5d} ({cnt/len(train_claims):.1%})')

# 统计每条 claim 对应的 Ground Truth (GT) evidence 数量
gt_counts = [len(c['evidences']) for c in train_claims.values()]
print(f'\n每条 claim 的 GT evidence 数:')
print(f'    min={min(gt_counts)}, max={max(gt_counts)},')
print(f'    mean={np.mean(gt_counts):.2f}, median={int(np.median(gt_counts))}')

训练集标签分布:
    SUPPORTS               519 (42.3%)
    NOT_ENOUGH_INFO        386 (31.4%)
    REFUTES                199 (16.2%)
    DISPUTED               124 (10.1%)

每条 claim 的 GT evidence 数:
    min=1, max=5,
    mean=3.36, median=3


In [ ]:
def eval_retrieval(claims_dataset, predict):
    all_recalls, all_precisions, all_fscores = [], [], []

    for claim_id, claim in sorted(claims_dataset.items()):
        if claim_id not in predict:
            continue

        evidence_correct = 0
        evidence_recall = 0.0
        evidence_precision = 0.0
        evidence_fscore = 0.0

        # 确保预测结果存在且不为空
        if isinstance(predict[claim_id], list) and len(predict[claim_id]) > 0:
            predict_set = set(predict[claim_id])

            # 计算预测对的证据数量（交集）
            for true_eid in claim['evidences']:
                if true_eid in predict_set:
                    evidence_correct += 1

            if evidence_correct > 0:
                # Recall: 找回了多少比例的正确证据
                evidence_recall = evidence_correct / len(claim['evidences'])
                # Precision: 预测出的证据中有多少是真正正确的
                evidence_precision = evidence_correct / len(predict[claim_id])
                # F1: 精确率和召回率的调和平均
                evidence_fscore = (2 * evidence_precision * evidence_recall) / (evidence_precision + evidence_recall)

        all_recalls.append(evidence_recall)
        all_precisions.append(evidence_precision)
        all_fscores.append(evidence_fscore)

    print(f'Mean Recall:    {np.mean(all_recalls):.6f}')
    print(f'Mean Precision: {np.mean(all_precisions):.6f}')
    print(f'Mean F1-Score:  {np.mean(all_fscores):.6f}')

    return np.mean(all_fscores)

In [ ]:
# 安装 bm25s 库和句子转换器
!pip install -q bm25s sentence-transformers

In [ ]:
import time
import bm25s
import psutil

# 提取证据库的 ID 和文本
evidence_ids    = list(evidence.keys())
evidence_texts  = [evidence[eid] for eid in evidence_ids]

print('Tokenizing 1.2M evidence ...')
t0 = time.time()

# 对证据进行分词处理（Tokenization）
# 使用英文停用词过滤，暂不使用词干提取（stemmer=None）
corpus_tokens = bm25s.tokenize(evidence_texts, stopwords='en', stemmer=None)

print(f'  tokenize 用时: {time.time() - t0:.0f}s')

print('\nBuilding BM25 index ...')
t0 = time.time()

# 初始化 BM25 检索器并构建索引
retriever = bm25s.BM25()
retriever.index(corpus_tokens)

print(f'  index 用时: {time.time() - t0:.0f}s')

# base1用的gpu，bm25用的是cpu

In [ ]:
from tqdm.auto import tqdm

def bm25s_retrieve(claims_dict, k=100):
    # 提取所有 claim 的 ID 和文本内容
    cids  = list(claims_dict.keys())
    texts = [claims_dict[c]['claim_text'] for c in cids]

    # 对查询（claims）进行分词处理
    query_tokens = bm25s.tokenize(texts, stopwords='en', stemmer=None)
    # 和之前的分词要相同

    # 在之前构建好的索引（retriever）中进行检索
    # results 会返回匹配到的证据索引，scores 返回相似度分值
    results, scores = retriever.retrieve(query_tokens, k=k)

    out = {}
    for i, cid in enumerate(cids):
        # 根据返回的索引 j，从证据 ID 列表（evidence_ids）中找回原始 ID
        out[cid] = [evidence_ids[j] for j in results[i]]

    return out

In [ ]:
print("BM25s baseline(top-5)")
t0 = time.time()
TOP_K = 5

# 执行检索，这里的 TOP_K 沿用你之前设置的 5
predict_R2 = bm25s_retrieve(dev_claims, k=TOP_K)

print(f'  用时: {time.time() - t0:.1f}s\n')

# 评估 BM25 的检索效果
f_R2 = eval_retrieval(dev_claims, predict_R2)

BM25s baseline(top-5)


Split strings:   0%|          | 0/154 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/154 [00:00<?, ?it/s]

  用时: 0.7s

Mean Recall:    0.160281
Mean Precision: 0.089610
Mean F1-Score:  0.107689


# 优化bm25s的逻辑，使用sbert+加权

In [ ]:

import torch
import time
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer, util

# 1. 定义设备（解决报错的关键点）
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. 加载 SBERT 预训练模型 (all-MiniLM-L6-v2) 并移动到 GPU
print("Loading SBERT rerank model...")
rerank_model = SentenceTransformer('all-MiniLM-L6-v2').to(device)
# --- 步骤 2: 定义混合检索重排序函数 ---
def hybrid_retrieve(claims_dict, k_coarse=50, k_fine=5, alpha=0.7):
    """
    k_coarse: 粗排召回数量 (建议 50)
    k_fine: 最终选出的证据数量 (比赛要求 5)
    alpha: 权重系数 (0.5 为平衡点)
    """
    cids = list(claims_dict.keys())
    texts = [claims_dict[c]['claim_text'] for c in cids]

    # 1. BM25 粗排 (Recall)
    query_tokens = bm25s.tokenize(texts, stopwords='en', stemmer=None)
    results, bm25_scores = retriever.retrieve(query_tokens, k=k_coarse)

    out = {}
    print(f"Reranking {len(cids)} claims...")

    # 循环处理每个 claim 进行精排
    for i, cid in enumerate(tqdm(cids)):
        candidate_indices = results[i]
        candidate_eids = [evidence_ids[idx] for idx in candidate_indices]
        candidate_texts = [evidence[eid] for eid in candidate_eids]

        # 2. SBERT 精排 (Reranking)
        claim_emb = rerank_model.encode(texts[i], convert_to_tensor=True)
        ev_embs = rerank_model.encode(candidate_texts, convert_to_tensor=True)

        # 计算 Claim 与候选证据的余弦相似度
        semantic_scores = util.cos_sim(claim_emb, ev_embs)[0]

        # 3. 加权融合 (Weighted Score)
        # 归一化 BM25 分数
        max_b25 = bm25_scores[i].max() if bm25_scores[i].max() > 0 else 1.0
        norm_bm25 = torch.tensor(bm25_scores[i] / max_b25).to(device)

        # 融合公式: alpha * BM25 + (1 - alpha) * SBERT
        final_scores = alpha * norm_bm25 + (1 - alpha) * semantic_scores

        # 选取最终得分最高的前 k_fine (5) 条
        top_k_idx = torch.topk(final_scores, k=k_fine).indices.cpu().numpy()
        out[cid] = [candidate_eids[idx] for idx in top_k_idx]

    return out

# # --- 步骤 3: 在开发集上执行并评估优化后的效果 ---
# print("\n[Hybrid Retrieval] Running Rerank on Dev Set...")
# t_start = time.time()
# # 得到优化后的检索结果
# predict_R2_hybrid = hybrid_retrieve(dev_claims, k_coarse=50, k_fine=5, alpha=0.3)

# print(f'Hybrid 检索用时: {time.time() - t_start:.1f}s\n')

# # 调用你之前的评估函数，对比 F1-Score 是否有显著提升
# f_R2_hybrid = eval_retrieval(dev_claims, predict_R2_hybrid)

# --- 步骤 3: 分别生成训练集和开发集的检索结果 ---

# 3.1 为训练集生成结果 (用于训练 model_C2_v2)
print("\n[Hybrid Retrieval] Generating Rerank for Training Set...")
predict_train_hybrid = hybrid_retrieve(train_claims, alpha=0.3)

# 3.2 为开发集生成结果 (用于评估和验证)
print("\n[Hybrid Retrieval] Running Rerank on Dev Set...")
predict_dev_hybrid = hybrid_retrieve(dev_claims, alpha=0.3)

# 评估开发集
f_R2_hybrid = eval_retrieval(dev_claims, predict_dev_hybrid)

In [ ]:
# import gc, psutil

# # 删除不再需要的巨型变量，以释放内存空间
# del cos_word       # 删除 TF-IDF 的相似度矩阵
# del corpus_tokens  # 删除 BM25 分词后的语料库
# del retriever      # 删除 BM25 检索器对象

# # 显式调用垃圾回收器，立即清理被删除变量占用的内存
# gc.collect()

# classification

# BiLSTM

# DistilBERT（BERT 的轻量化版本

In [ ]:
# # 1. 安装 Hugging Face 的 transformers 库
# !pip install -q transformers

# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# from transformers import get_linear_schedule_with_warmup

# # 2. 选择预训练模型：这里使用的是 DistilBERT（BERT 的轻量化版本）
# # 它在保持高性能的同时，比 BERT 快 60%，体积小 40%
# MODEL_NAME = 'distilbert-base-uncased'

# # 3. 加载分词器和模型
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# # 加载专门用于序列分类的模型头
# model_C2_v2  = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_NAME,
#     num_labels=4,            # 对应你的 4 个标签
#     id2label=ID2LABEL,      # 之前定义的 ID 到标签映射
#     label2id=LABEL2ID,      # 之前定义的标签 到 ID 映射
# ).to(device)

# # 4. 计算并打印参数量
# n_params = sum(p.numel() for p in model_C2.parameters())
# print(f'DistilBERT 参数量: {n_params:,}')


In [ ]:
# 1. 确保安装 transformers 库
!pip install -q transformers

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 2. 依然选择 DistilBERT 作为基础模型
MODEL_NAME = 'distilbert-base-uncased'

# 3. 加载分词器和新的模型实例
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# --- 建议修改点：变量名改为 model_C2_v2，表示这是使用混合检索证据训练的版本 ---
model_C2_v2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),  # 对应 4 个标签
    id2label=ID2LABEL,
    label2id=LABEL2ID,
).to(device) # 确保移动到 GPU

# 4. 打印参数量确认
n_params = sum(p.numel() for p in model_C2_v2.parameters())
print(f'New DistilBERT (v2) 参数量: {n_params:,}')

In [ ]:
# distrilBERT比LSTM 参数量很多？考虑过拟合的问题？
class ClaimEvDatasetBERT(Dataset):
    """
    和 LSTM 版结构一样，但用 HuggingFace tokenizer 编码，不再自己拼 [SEP]。
    Tokenizer 会在 claim 和 evidence 中间自动插入 [SEP] token。
    """
    def __init__(self, claims_dict, evidence_dict, retrieval=None,
                 tokenizer=None, max_len=512):
        self.cids      = list(claims_dict.keys())
        self.claims    = claims_dict
        self.evidence  = evidence_dict
        self.retrieval = retrieval
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.has_label = 'claim_label' in next(iter(claims_dict.values()))

    def __len__(self):
        return len(self.cids)

    def __getitem__(self, i):
        cid = self.cids[i]
        c   = self.claims[cid]

        # 确定证据 ID 列表：训练用真实答案，开发/测试用检索结果
        ev_ids = c['evidences'] if self.retrieval is None else self.retrieval[cid]
        # 将前 5 条证据文本拼接成一个长字符串
        ev_text = ' '.join([self.evidence[e] for e in ev_ids[:5]])

        # 使用 BERT Tokenizer 进行编码
        enc = self.tokenizer(
            c['claim_text'],          # 第一个句子
            ev_text,                  # 第二个句子
            truncation=True,          # 超出长度自动截断
            padding='max_length',     # 长度不足自动填充
            max_length=self.max_len,  # 最大长度（256）
            return_tensors='pt',      # 返回 PyTorch 张量
        )

        # 移除多余的维度并转化为字典格式
        item = {k: v.squeeze(0) for k, v in enc.items()}

        if self.has_label:
            # 添加标签
            item['labels'] = torch.tensor(LABEL2ID[c['claim_label']], dtype=torch.long)

        return item

# # 实例化数据集
# train_ds_bert = ClaimEvDatasetBERT(train_claims, evidence, retrieval=None, tokenizer=tokenizer, max_len=512)
# # 注意：这里开发集传入了 predict_R2，也就是你之前用 BM25 跑出来的检索结果
# dev_ds_bert   = ClaimEvDatasetBERT(dev_claims,   evidence, retrieval=predict_R2, tokenizer=tokenizer, max_len=512)

# 1. 训练集实例化更新：
# 关键修改：将 retrieval 从 None 改为你用混合检索跑出的训练集结果 predict_train_hybrid
# 这样模型在训练时看到的就是 SBERT 精排后的 5 条证据，而不是 Ground Truth 完美证据
train_ds_bert = ClaimEvDatasetBERT(
    train_claims,
    evidence,
    retrieval=predict_train_hybrid,  # 确保你已经跑完了训练集的 hybrid_retrieve
    tokenizer=tokenizer,
    max_len=512
)
# 2. 开发集实例化更新：
# 关键修改：将 retrieval 从旧的 predict_R2 (BM25) 改为优化后的 predict_dev_hybrid (SBERT精排)
# 这样你的本地评估结果（Dev Acc）才能真实反映你在排行榜上的表现
dev_ds_bert = ClaimEvDatasetBERT(
    dev_claims,
    evidence,
    retrieval=predict_dev_hybrid,    # 使用最优 alpha=0.3 跑出的开发集检索结果
    tokenizer=tokenizer,
    max_len=512
)

# 实例化 DataLoader
# BERT 模型较大，batch_size 通常设小一点（比如 16）以防显存溢出
train_loader_bert = DataLoader(train_ds_bert, batch_size=16, shuffle=True)
dev_loader_bert   = DataLoader(dev_ds_bert,   batch_size=32, shuffle=False)

In [ ]:
#名字/顺序换一下，基本都会这样写。'
# 运行不出来

In [ ]:
# 1. 设置微调参数
EPOCHS_BERT = 6
LR_BERT     = 2e-5  # 预训练模型通常使用非常小的学习率

# 使用 AdamW 优化器，它比普通 Adam 更适合微调 Transformer 模型，包含了权重衰减 (Weight Decay)
optimizer_bert = torch.optim.AdamW(model_C2_v2.parameters(), lr=LR_BERT, weight_decay=0.01)

# 计算总训练步数，用于设置学习率调度器
total_steps    = len(train_loader_bert) * EPOCHS_BERT

# 2. 设置学习率调度器 (Scheduler)
# 包含 Warmup 阶段：学习率在前 10% 的步数中从 0 增加到 2e-5，然后线性下降。这能防止模型在训练初期崩溃。
scheduler_bert = get_linear_schedule_with_warmup(
    optimizer_bert,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# 沿用之前的类别权重损失函数
loss_fn_bert = nn.CrossEntropyLoss(weight=class_weights)

# 3. 正式开始训练
for epoch in range(EPOCHS_BERT):
    model_C2_v2.train()
    epoch_losses = []

    for batch in train_loader_bert:
        # 将整个 batch 的数据（input_ids, attention_mask 等）搬运到 GPU
        batch  = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop('labels')  # 弹出标签用于计算 loss

        # 前向传播：**batch 会将字典解包为模型的参数输入
        outputs = model_C2_v2(**batch)
        logits  = outputs.logits
        loss    = loss_fn_bert(logits, labels)

        # 反向传播
        optimizer_bert.zero_grad()
        loss.backward()

        # 梯度裁剪：Transformer 建议设为 1.0
        torch.nn.utils.clip_grad_norm_(model_C2_v2.parameters(), max_norm=1.0)

        optimizer_bert.step()   # 更新参数
        scheduler_bert.step()   # 更新学习率

        epoch_losses.append(loss.item())

    print(f'Epoch {epoch+1}/{EPOCHS_BERT}  train_loss = {np.mean(epoch_losses):.4f}')
    # --- 建议增加：每轮跑一次开发集验证 ---
    # 这能让你立刻看到检索感知训练是否提升了 Accuracy
    dev_results = predict_with_bert(model_C2_v2, dev_loader_bert, dev_ds_bert)
    dev_acc = sum(1 for cid, lbl in dev_results.items() if dev_claims[cid]['claim_label'] == lbl) / len(dev_results)
    print(f'>> Dev Accuracy: {dev_acc:.4f}')

In [ ]:
def predict_with_bert(model, data_loader, dataset):
    """
    跑 transformer 拿到每条 claim 的预测标签。
    """
    model.eval()  # 切换到评估模式，关闭 Dropout
    all_preds = []

    with torch.no_grad():  # 禁用梯度计算，节省显存
        for batch in data_loader:
            # 移除标签（如果存在），因为预测阶段不需要它
            batch.pop('labels', None)
            # 将数据移动到 GPU
            batch = {k: v.to(device) for k, v in batch.items()}

            # 前向传播：解包 batch 字典并获取 logits
            logits = model(**batch).logits
            # 获取得分最高的类别索引
            preds  = logits.argmax(dim=-1).cpu().tolist()
            all_preds.extend(preds)

    # 将结果映射回字典：{claim_id: label_string}
    return {dataset.cids[i]: ID2LABEL[p] for i, p in enumerate(all_preds)}

# 1. 运行预测
pred_label_C2 = predict_with_bert(model_C2_v2, dev_loader_bert, dev_ds_bert)

# 2. 计算正确预测的数量
correct = sum(1 for cid, plabel in pred_label_C2.items()
             if dev_claims[cid]['claim_label'] == plabel)

# 3. 计算并打印准确率
acc_C2 = correct / len(pred_label_C2)
print(f'C2 Classification Accuracy on dev: {acc_C2:.4f}')

# 输出test

# bm25+BiLSTM

In [ ]:
# import json

# # 1. 使用 BM25 为测试集检索 Top-5 证据
# print("Retrieving evidence for test set using BM25...")
# # 这里的 test_claims 是你之前通过 json.load 加载的 test-claims-unlabelled.json
# predict_test_retrieval = bm25s_retrieve(test_claims, k=5)

# # 2. 准备测试集 Dataset 和 DataLoader
# # 注意：这里传入 retrieval=predict_test_retrieval 使用刚才检索到的证据
# test_ds_lstm = ClaimEvDatasetLSTM(
#     test_claims,
#     evidence,
#     retrieval=predict_test_retrieval,
#     max_len=256
# )
# test_loader_lstm = DataLoader(test_ds_lstm, batch_size=32, shuffle=False)

# # 3. 使用训练好的 BiLSTM 模型 (model_C1) 进行分类预测
# print("Predicting labels for test set using BiLSTM...")
# # 使用你定义的 predict_with_lstm 函数
# pred_label_test = predict_with_lstm(model_C1, test_loader_lstm, test_ds_lstm)

# # 4. 按照比赛要求的格式整合输出
# # 格式要求：{ "claim_id": { "claim_label": "...", "evidences": ["...", "..."] } }
# test_output = {}
# for cid in test_claims.keys():
#     test_output[cid] = {
#         "claim_label": pred_label_test[cid],
#         "evidences": predict_test_retrieval[cid]
#     }

# # 5. 保存为 json 文件
# output_path = 'test-output.json'
# with open(output_path, 'w', encoding='utf-8') as f:
#     json.dump(test_output, f, indent=4)

# print(f"Success! Test output saved to {output_path}")

# SBERT+BiLSTM

In [ ]:
# # SBERT 精排+ BiLSTM
# import json

# # 1. 使用混合检索 (Hybrid Retrieval) 为测试集筛选证据
# # 相比基础 BM25，这一步引入了 SBERT 语义重排序，能显著提升检索质量
# print("Running Hybrid Retrieval for test set (BM25 + SBERT)...")
# # k_coarse=50 保证召回率，k_fine=5 保证精排准确度
# predict_test_hybrid = hybrid_retrieve(test_claims, k_coarse=50, k_fine=5, alpha=0.5)

# # 2. 准备测试集 Dataset 和 DataLoader
# # 使用混合检索得到的 predict_test_hybrid 作为输入
# test_ds_final = ClaimEvDatasetLSTM(
#     test_claims,
#     evidence,
#     retrieval=predict_test_hybrid,
#     max_len=256 # 保持与模型训练时的长度一致
# )
# test_loader_final = DataLoader(test_ds_final, batch_size=32, shuffle=False)

# # 3. 使用训练好的 BiLSTM 模型 (model_C1) 进行分类预测
# print("Predicting labels for test set using BiLSTM...")
# # 模型将基于更高质量的语义证据进行判断，有助于提升分类准确率
# pred_label_test_final = predict_with_lstm(model_C1, test_loader_final, test_ds_final)

# # 4. 按照比赛要求的格式整合输出
# test_output_final = {}
# for cid in test_claims.keys():
#     test_output_final[cid] = {
#         "claim_label": pred_label_test_final[cid],
#         "evidences": predict_test_hybrid[cid]
#     }

# # 5. 保存为 json 文件
# output_path = 'test-output.json'
# with open(output_path, 'w', encoding='utf-8') as f:
#     json.dump(test_output_final, f, indent=4)

# print(f"\nSuccess! Hybrid output saved to {output_path}")

# SBERT+DistilBERT



In [ ]:
import json
import torch
from torch.utils.data import DataLoader

# --- 步骤 1: 使用混合检索 (Hybrid Retrieval) 筛选证据 ---
# 建议使用你在 Dev 集上调试出的最佳 alpha 值（例如 0.2 或 0.3）
BEST_ALPHA = 0.3
print(f"Running Hybrid Retrieval for test set (BM25 + SBERT, alpha={BEST_ALPHA})...")

# 使用你之前定义的 hybrid_retrieve 函数
predict_test_hybrid = hybrid_retrieve(test_claims, k_coarse=50, k_fine=5, alpha=BEST_ALPHA)

# --- 步骤 2: 准备基于 BERT 的测试集和加载器 ---
# 注意：这里必须使用 ClaimEvDatasetBERT 类，因为它包含了 BERT 的 Tokenizer 逻辑
test_ds_bert_final = ClaimEvDatasetBERT(
    test_claims,
    evidence,
    retrieval=predict_test_hybrid,
    tokenizer=tokenizer,  # 使用之前加载的 distilbert-base-uncased tokenizer
    max_len=512           # DistilBERT 建议使用 512 以包含更多上下文信息
)

# BERT 模型较大，batch_size 建议设为 32 或更小
test_loader_bert_final = DataLoader(test_ds_bert_final, batch_size=32, shuffle=False)

# --- 步骤 3: 使用微调后的 DistilBERT (model_C2_v2) 进行预测 ---
print("Predicting labels for test set using DistilBERT...")
# 使用你定义的 predict_with_bert 函数
pred_label_test_bert = predict_with_bert(model_C2_v2, test_loader_bert_final, test_ds_bert_final)

# --- 步骤 4: 整合并保存输出结果 ---
test_output_bert = {}
for cid in test_claims.keys():
    test_output_bert[cid] = {
        "claim_label": pred_label_test_bert[cid],
        "evidences": predict_test_hybrid[cid]
    }

# 保存文件
output_path = 'test-output.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(test_output_bert, f, indent=4)

print(f"\nSuccess! DistilBERT + Hybrid output saved to {output_path}")

# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*